In [ ]:
import sys
sys.path.append("..")

In [ ]:
import torch
import torch.nn.functional as F
import pandas as pd
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from pathlib import Path
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

from src.model import CardinalityEstimator
from src.dataset import GalacticBinariesDataset
from config import val_dataset_path, BATCH_SIZE, MAX_K

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

ckpt_path = Path("../runs/run_20260511_120831/checkpoints/best_checkpoint.pth")

model = CardinalityEstimator(learning_strategy="mse", max_K=10, input_channels=4).to(device)
ckpt = torch.load(ckpt_path, map_location=device)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()

In [ ]:
NB_RUNS= 10
seeds = [0, 2026, 45, 1234, 5678, 91011, 121314, 151617, 181920, 212223]
all_results = []

for run in range(NB_RUNS):
    val_ds = GalacticBinariesDataset(
        val_dataset_path,
        max_K=10,
        max_samples=100_000,
        noise=True,
        deterministic=True,
        energy_matched=True,
        target_energy=22000.0,
        seed=seeds[run],
    )

    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

    rows = []

    with torch.no_grad():
        offset = 0
        for summed_waveform, targets in val_loader:
            X = torch.as_tensor(summed_waveform, dtype=torch.float32, device=device)
            targets = torch.as_tensor(targets, dtype=torch.long, device=device)
            y = targets.float().unsqueeze(1) / MAX_K

            logit = model(X)
            out = torch.sigmoid(logit)

            pred = torch.round(out * MAX_K).long().squeeze(1)
            nll = F.binary_cross_entropy(out, y, reduction="none").squeeze(1)

            for i in range(len(y)):
                rows.append({
                    "idx": offset + i,
                    "true_class": y[i].item(),
                    "pred_class": out[i].item(),
                    "true_K": targets[i].item(),
                    "pred_K": pred[i].item(),
                    "correct": pred[i].item() == targets[i].item(),
                    "error": out[i].item() - y[i].item(),
                    "abs_error": abs(pred[i].item() - targets[i].item()),
                    "nll": nll[i].item(),
                })

            offset += len(y)

    df = pd.DataFrame(rows)
    all_results.append(df)

In [ ]:
final_df = pd.concat(all_results, ignore_index=True)
print("Average MSE:", (final_df["error"] ** 2).mean())
print("Average Precision:", final_df["correct"].mean())

In [ ]:
summary_by_k = (
    final_df.groupby("true_K")
    .agg(
        n=("idx", "count"),
        accuracy=("correct", "mean"),
        mae=("abs_error", "mean"),
        error=("error", "mean"),
        avg_nll=("nll", "mean"),
    )
)

summary_by_k

In [ ]:
correct_counts = summary_by_k.accuracy * summary_by_k.n
print(correct_counts)

plt.figure(figsize=(10, 6))
plt.plot(summary_by_k.index, summary_by_k.accuracy * summary_by_k.n, marker="o")
plt.xlabel("True K")
plt.ylabel("Number of correct predictions")
plt.title("Number of correct predictions by true K")
plt.xticks(summary_by_k.index)
plt.grid(axis="y")
plt.show()

In [ ]:
cm = confusion_matrix(final_df.true_K, final_df.pred_K, labels=range(1, 11))
disp = ConfusionMatrixDisplay(cm, display_labels=range(1, 11))
disp.plot(cmap="Blues", values_format="d")
plt.xlabel("Predicted K")
plt.ylabel("True K")
plt.show()

In [ ]:
df["abs_error"].value_counts().sort_index().plot(kind="bar")
print(df["abs_error"].value_counts().sort_index())
plt.xlabel("|predicted K - true K|")
plt.ylabel("Number of examples")
plt.title("Error magnitude distribution")
plt.show()